# 이슈 #38 구조환경지표 원자료 기반 전수검증

## 목적과 현재 범위

- `configs/structural_indicators_verification.yaml`을 기준으로 구조환경지표 21개의 가공값과 원자료 연결 상태를 점검한다.
- 현재 상세 검증이 완료된 지표는 `2-1.1. 보육시설 보급률`이다. 나머지 지표의 산식 검증은 후속 작업이다.
- 기존 가공값은 `어린이집 수 ÷ 0–4세 인구 × 100`과 전부 일치했으며, 명세상 올바른 산식은 `어린이집 정원 ÷ 0–4세 인구 × 100`이다.
- 이 노트북은 수정값을 메모리에 생성할 뿐 원본 또는 기존 가공 파일을 덮어쓰지 않는다.

## 1. 실행 환경과 검증 명세 확인

저장소 루트를 찾고 필수 라이브러리를 불러온 뒤, 검증 명세와 구조환경 원자료 경로가 존재하는지 확인한다. YAML의 필수 구역, 지표 수, 지표 ID 중복 여부도 검사한다. 정상 출력은 이슈 `#38`, 검증 대상 `21개`, 고유 ID `21개`이다.

In [ ]:
import hashlib
import re
import unicodedata

import openpyxl
import pandas as pd
import yaml

from pathlib import Path

repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / ".git").exists()), None
)

if repo_root is None:
    raise FileNotFoundError("현재 실행 위치의 상위 경로에서 Git 저장소를 찾지 못했습니다.")

print(f"저장소 루트: {repo_root}")

## 1-1. 공통 설정 - 지역명 매핑

이 노트북 전체(모든 지표 검증)에서 지역명 정규화는 이 `region_order`/`region_map` 하나만 쓴다. 시도 축약형·전체 행정구역명(예: 서울특별시)·옛 명칭(강원도 등)·"전체"/"계" 같은 총계 표기까지 전부 포함한다.

In [ ]:
lookup_path = repo_root / "data" / "lookup" / "시도_지역코드_매핑.csv"
region_lookup = pd.read_csv(lookup_path)

region_order = ["전국", *region_lookup["지역"].tolist()]

# 룩업 파일은 각 지역의 "현재" 공식 전체명만 담고 있어, 일부 원자료가 쓰는 옛
# 행정구역명(강원도 등)과 "전국"의 여러 표기(전체/계)는 별도로 추가한다.
LEGACY_REGION_NAMES = {
    "강원도": "강원",
    "전라북도": "전북",
    "제주도": "제주",
}
NATIONWIDE_ALIASES = {"전체": "전국", "계": "전국"}

region_map = (
    {r: r for r in region_order}
    | dict(zip(region_lookup["지역명_전체"], region_lookup["지역"]))
    | LEGACY_REGION_NAMES
    | NATIONWIDE_ALIASES
)

print(f"region_order: {len(region_order)}개 지역 (룩업 {len(region_lookup)}개 시도 + 전국)")

In [9]:
config_path = repo_root / "configs" / "structural_indicators_verification.yaml"
raw_root = repo_root / "data" / "raw"

missing_paths = [path for path in (config_path, raw_root) if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "다음 입력 경로를 찾지 못했습니다:\n" + "\n".join(map(str, missing_paths))
    )

print(f"검증 명세: {config_path.relative_to(repo_root)}")
print(f"구조환경 원자료: {raw_root.relative_to(repo_root)}")

검증 명세: configs/structural_indicators_verification.yaml
구조환경 원자료: data/raw


In [10]:
with config_path.open(encoding="utf-8") as file:
    manifest = yaml.safe_load(file)

required_sections = {"manifest_version", "issue", "scope", "rules", "local", "drive", "indicators"}
missing_sections = required_sections - manifest.keys()
if missing_sections:
    raise KeyError(f"YAML 필수 구역이 없습니다: {sorted(missing_sections)}")

indicators = manifest["indicators"]
if not isinstance(indicators, list) or not all(
    isinstance(indicator, dict) for indicator in indicators
):
    raise TypeError("indicators는 지표 딕셔너리로 구성된 리스트여야 합니다.")

indicator_ids = [indicator.get("id") for indicator in indicators]
if len(indicators) != manifest["scope"]["completed_indicators_to_validate"]:
    raise ValueError("검증 대상 지표 수가 scope의 선언값과 다릅니다.")
if None in indicator_ids or len(indicator_ids) != len(set(indicator_ids)):
    raise ValueError("지표 ID가 없거나 중복됐습니다.")

print(
    f"명세 버전: {manifest['manifest_version']} | 이슈: #{manifest['issue']} | "
    f"검증 대상: {len(indicators)}개 | 고유 ID: {len(set(indicator_ids))}개"
)

명세 버전: 3 | 이슈: #38 | 검증 대상: 21개 | 고유 ID: 21개


## 2. 명세와 로컬 원자료 연결 확인

YAML에 기록된 가공 파일·원자료 파일명을 로컬 원자료 폴더와 연결한다. 같은 이름의 파일이 여러 개면 SHA-256 해시로 실제 내용까지 비교한다. 현재 누락 파일은 없지만, 주민등록인구 파일은 같은 이름의 서로 다른 내용이 있어 지표에 맞는 파일을 명시적으로 선택해야 한다.

In [ ]:
def normalize_name(name: str) -> str:
    return unicodedata.normalize("NFC", name)


def core_key(name: str) -> str:
    # 파일명 자체가 아니라 "언제 재다운로드했는지"만 다른 경우를 찾기 위한 느슨한 키.
    # 8자리 날짜(예: 20181231)는 스냅샷 연도를 구분하는 의미가 있어 건드리지 않고,
    # 14자리 다운로드 시각(예: 20260715191848)과 맨 앞 연도범위·괄호주석만 제거한다.
    key = normalize_name(name)
    key = re.sub(r"^\([^)]*\)_", "", key)
    key = re.sub(r"^\d{4}(-\d{4})?_", "", key)
    key = re.sub(r"_\d{14}(?=\.[^.]+$)", "", key)
    return key


def extension_variants(name: str) -> set[str]:
    # 드라이브가 CSV를 엑셀로 다시 저장하면서 이름이 "name.csv.xlsx"로 늘어나거나
    # ".csv"가 통째로 ".xlsx"로 바뀌는 경우가 있어, 두 변형도 같이 찾아본다.
    variants = {name}
    if name.lower().endswith(".csv"):
        variants.add(name + ".xlsx")
        variants.add(name[:-4] + ".xlsx")
    return variants


def find_matches(paths_by_name: dict[str, list[Path]], file_name: str) -> list[Path]:
    for candidate in extension_variants(normalize_name(file_name)):
        matched = paths_by_name.get(candidate)
        if matched:
            return matched
    return []


local_files: list[Path] = sorted(
    (path for path in raw_root.rglob("*") if path.is_file()),
    key=lambda path: path.as_posix().casefold(),
)

if not local_files:
    raise FileNotFoundError(f"원자료 폴더가 비어 있습니다: {raw_root}")

paths_by_name: dict[str, list[Path]] = {}
paths_by_core_key: dict[str, list[Path]] = {}

for path in local_files:
    paths_by_name.setdefault(normalize_name(path.name), []).append(path)
    paths_by_core_key.setdefault(core_key(path.name), []).append(path)

# 재정팀 산출값 파일 안에 원자료가 시트로 이미 들어있는지 확인한다.
# 시트가 2개 이상이면 계산 시트·원자료 시트가 포함된 것으로 본다.
# 보육시설 보급률(numerator_active, 정원 데이터)만 예외로, 파일 안에 없는 게 확인됐다.
EMBEDDED_ROLE_EXCEPTIONS = {("childcare_capacity_rate", "numerator_active")}

# YAML rules.no_separate_raw_data_indicators에 등록된, 애초에 별도 원자료가 없는 지표.
NO_RAW_DATA_INDICATORS = set(manifest["rules"].get("no_separate_raw_data_indicators", []))

sheet_counts_by_indicator: dict[str, int] = {}

for indicator in indicators:
    matched = find_matches(paths_by_name, indicator["finance_team_value"]["file_name"])
    if not matched:
        continue

    workbook = openpyxl.load_workbook(matched[0], read_only=True)
    sheet_counts_by_indicator[indicator["id"]] = len(workbook.sheetnames)
    workbook.close()

mapping_records: list[dict[str, str]] = []

for indicator in indicators:
    mapping_records.append(
        {
            "indicator_id": indicator["id"],
            "mapping_type": "finance_team_value",
            "role": "finance_team_value",
            "drive_id": indicator["finance_team_value"]["id"],
            "file_name": indicator["finance_team_value"]["file_name"],
            "embedded_in_finance_team_value": False,
            "no_raw_data_expected": False,
        }
    )

    for source in indicator["raw_sources"]:
        embedded = (
            sheet_counts_by_indicator.get(indicator["id"], 1) > 1
            and (indicator["id"], source["role"]) not in EMBEDDED_ROLE_EXCEPTIONS
        )
        mapping_records.append(
            {
                "indicator_id": indicator["id"],
                "mapping_type": "raw",
                "role": source["role"],
                "drive_id": source["id"],
                "file_name": source["file_name"],
                "embedded_in_finance_team_value": embedded,
                "no_raw_data_expected": indicator["id"] in NO_RAW_DATA_INDICATORS,
            }
        )

resolution_rows: list[dict[str, object]] = []

for record in mapping_records:
    matched_paths = find_matches(paths_by_name, record["file_name"])
    candidate_paths = (
        [] if matched_paths else paths_by_core_key.get(core_key(record["file_name"]), [])
    )

    resolution_rows.append(
        {
            **record,
            "match_count": len(matched_paths),
            "local_paths": " | ".join(str(path.relative_to(repo_root)) for path in matched_paths),
            "timestamp_candidate_count": len(candidate_paths),
            "timestamp_candidate_paths": " | ".join(
                str(path.relative_to(repo_root)) for path in candidate_paths
            ),
        }
    )

file_resolution = pd.DataFrame(resolution_rows)
file_resolution["embedded_in_finance_team_value"] = file_resolution[
    "embedded_in_finance_team_value"
].astype(bool)
file_resolution["no_raw_data_expected"] = file_resolution["no_raw_data_expected"].astype(bool)

still_needed = file_resolution[
    ~file_resolution["embedded_in_finance_team_value"] & ~file_resolution["no_raw_data_expected"]
].copy()

no_candidate = still_needed[
    (still_needed["match_count"] == 0) & (still_needed["timestamp_candidate_count"] == 0)
].copy()

timestamp_candidates = still_needed[
    (still_needed["match_count"] == 0) & (still_needed["timestamp_candidate_count"] > 0)
].copy()

ambiguous_file_mappings = still_needed[still_needed["match_count"] > 1].copy()

embedded_count = file_resolution["embedded_in_finance_team_value"].sum()
no_raw_data_count = file_resolution["no_raw_data_expected"].sum()

print(
    f"로컬 탐색 파일: {len(local_files)}개 | "
    f"매핑 참조: {len(file_resolution)}건 | "
    f"파일 안에 이미 있음(임베디드): {embedded_count}건 | "
    f"원자료 없음(정상): {no_raw_data_count}건 | "
    f"정확히 1개 일치: {(still_needed['match_count'] == 1).sum()}건 | "
    f"타임스탬프만 다른 후보: {len(timestamp_candidates)}건 | "
    f"진짜 누락(후보 없음): {len(no_candidate)}건 | "
    f"다중 일치: {len(ambiguous_file_mappings)}건"
)

with pd.option_context("display.max_colwidth", None):
    if not timestamp_candidates.empty:
        print(f"\n타임스탬프만 다른 후보 발견 - 확인 필요 ({len(timestamp_candidates)}건):")
        display(
            timestamp_candidates[["indicator_id", "role", "file_name", "timestamp_candidate_paths"]]
            .sort_values(["indicator_id", "role"])
            .reset_index(drop=True)
        )

    if not no_candidate.empty:
        print(
            f"\n진짜 누락된 매핑 ({len(no_candidate)}건, 임베디드·원자료없음·타임스탬프후보 제외):"
        )
        display(
            no_candidate[["indicator_id", "role", "file_name"]]
            .sort_values(["indicator_id", "role"])
            .reset_index(drop=True)
        )

    if not ambiguous_file_mappings.empty:
        print(f"\n다중 일치 매핑 ({len(ambiguous_file_mappings)}건):")
        display(
            ambiguous_file_mappings[
                ["indicator_id", "role", "file_name", "match_count", "local_paths"]
            ]
            .sort_values(["indicator_id", "role"])
            .reset_index(drop=True)
        )

In [12]:
def calculate_sha256(path: Path) -> str:
    hasher = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            hasher.update(chunk)

    return hasher.hexdigest()


ambiguous_names = sorted(
    str(file_name) for file_name in ambiguous_file_mappings["file_name"].unique()
)

if not ambiguous_names:
    print("다중 일치 파일명이 없어 SHA-256 비교를 건너뜁니다.")
else:
    hash_rows: list[dict[str, object]] = []

    for file_name in ambiguous_names:
        for path in paths_by_name[normalize_name(file_name)]:
            hash_rows.append(
                {
                    "file_name": file_name,
                    "local_path": path.relative_to(raw_root).as_posix(),
                    "size_bytes": path.stat().st_size,
                    "sha256": calculate_sha256(path),
                }
            )

    duplicate_file_hashes = pd.DataFrame(hash_rows)

    duplicate_content_summary = duplicate_file_hashes.groupby("file_name", as_index=False).agg(
        candidate_count=("local_path", "size"),
        distinct_size_count=("size_bytes", "nunique"),
        distinct_hash_count=("sha256", "nunique"),
    )

    duplicate_content_summary["content_identical"] = (
        duplicate_content_summary["distinct_hash_count"] == 1
    )

    different_content_files = duplicate_content_summary[
        ~duplicate_content_summary["content_identical"]
    ].copy()

    print(
        f"중복 파일명: {len(duplicate_content_summary)}개 | "
        f"후보 파일: {len(duplicate_file_hashes)}개 | "
        f"내용 동일: {duplicate_content_summary['content_identical'].sum()}개 | "
        f"내용 다름: {len(different_content_files)}개"
    )

    if not different_content_files.empty:
        print("\n내용이 다른 중복 파일:")
        display(different_content_files)
        display(
            duplicate_file_hashes[
                duplicate_file_hashes["file_name"].isin(different_content_files["file_name"])
            ]
        )

다중 일치 파일명이 없어 SHA-256 비교를 건너뜁니다.


## 3. 보육시설 보급률 입력 파일 확정

- `finance_team_value`: 검증할 기존 보육시설 보급률
- `numerator_active`: 올바른 분자인 어린이집 정원 (아직 로컬에 없음 - 별도로 받아야 함)
- `numerator_excluded_facility_count`: 기존 오류 산식을 입증하기 위한 어린이집 수 비교자료이며, 수정 산식의 분자로 사용하지 않는다.

분모(0–4세 주민등록인구)는 별도 파일이 아니라 `finance_team_value`(`2-1.1. 보육시설 보급률.xlsx`)의 "0-4세 인구" 시트에 이미 포함돼 있어, 여기서는 따로 찾지 않는다.

In [13]:
# cell 9

childcare = next(item for item in indicators if item["id"] == "childcare_capacity_rate")

EMBEDDED_ROLES = {"denominator", "numerator_excluded_facility_count"}

childcare_paths = {
    "finance_team_value": paths_by_name[childcare["finance_team_value"]["file_name"]][0],
    **{
        source["role"]: paths_by_name[source["file_name"]][0]
        for source in childcare["raw_sources"]
        if source["role"] not in EMBEDDED_ROLES
    },
}

for role, path in childcare_paths.items():
    print(f"{role}: {path.relative_to(repo_root)}")

finance_team_value: data/raw/지표별_측정값/2-1.1. 보육시설 보급률.xlsx
numerator_active: data/raw/지표별_원데이터/2-1.1. 보육시설 보급률 원자료/전국_어린이집_정현원_현황_20260724113247.csv.xlsx


## 4. 입력 자료 구조 확인

선택한 엑셀 파일의 시트명, 행·열 크기, 상단 값을 미리 확인한다. 이 단계에서는 값을 변환하지 않으며, 2016–2025년 지역별 정원과 0–4세 인구가 실제로 들어 있는지 확인한다.

In [14]:
# cell 10

for role, path in childcare_paths.items():
    if path.suffix.lower() != ".xlsx":
        continue

    workbook = openpyxl.load_workbook(path, read_only=True, data_only=True)
    print(f"\n[{role}] {path.name}")

    for worksheet in workbook.worksheets:
        preview = list(worksheet.iter_rows(max_row=8, values_only=True))
        print(f"{worksheet.title}: {worksheet.max_row}행 × {worksheet.max_column}열")
        display(pd.DataFrame(preview))

    workbook.close()


[finance_team_value] 2-1.1. 보육시설 보급률.xlsx
2-1.1. 보육시설 보급률: 19행 × 12열


,0,1,2,3,4,5,6,7,8,9,10,11
0,지역,세부지표,2016.000000,2017.000000,2018.000000,2019.000000,2020.000000,2021.000000,2022.000000,2023.000000,2024.000000,2025.000000
1,전국,보육시설 보급률,1.863836,1.935343,1.984101,2.025395,2.108021,2.172275,2.166098,2.169508,2.153445,2.077639
2,서울,보육시설 보급률,1.691031,1.777450,1.851635,1.894288,1.985352,2.064710,2.070126,2.080145,2.075930,1.979006
3,부산,보육시설 보급률,1.466935,1.551139,1.614956,1.711698,1.825949,1.900248,1.903485,1.920117,1.903396,1.820668
4,대구,보육시설 보급률,1.494990,1.561200,1.576827,1.596483,1.708895,1.788809,1.860442,1.873687,1.886139,1.819249
5,인천,보육시설 보급률,1.711152,1.793656,1.855752,1.905036,2.017234,2.031458,1.983658,1.996302,1.981792,1.875218
6,광주,보육시설 보급률,1.893980,2.023730,2.061376,2.080398,2.198073,2.209531,2.211817,2.256141,2.294264,2.267541
7,대전,보육시설 보급률,2.322172,2.381291,2.396210,2.410901,2.485110,2.532723,2.434335,2.305505,2.258587,2.167126


보육시설 보급률 계산: 57행 × 13열


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,지역,항목,NaN,2016 년,2017 년,2018 년,2019 년,2020 년,2021 년,2022 년,2023 년,2024 년,2025 년
1,전국,총인구수[명],0-4세,2204271,2079115,1974244,1845122,1677023,1530469,1427590,1334588,1271776,1254501
2,서울,총인구수[명],0-4세,376575,350277,324470,300799,270481,244538,227619,213014,202897,202627
3,부산,총인구수[명],0-4세,132044,123780,117093,107963,97374,87778,81272,75360,71136,70194
4,대구,총인구수[명],0-4세,99198,93774,89103,82807,74200,66357,61222,57587,54874,54528
5,인천,총인구수[명],0-4세,130380,121874,115371,107557,96320,88754,85549,82753,81391,82977
6,광주,총인구수[명],0-4세,65365,61273,57971,53932,48770,45349,42499,38916,36090,34619
7,대전,총인구수[명],0-4세,68212,63201,58676,53424,47684,43471,41613,39731,38254,38115


0-4세 인구: 109행 × 13열


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,지역,연령별,항목,2016 년,2017 년,2018 년,2019 년,2020 년,2021 년,2022 년,2023 년,2024 년,2025 년
1,전국,0세,총인구수[명],393674,345786,317685,295132,265087,253946,244250,225958,235337,252212
2,전국,1세,총인구수[명],441720,409814,361625,330970,304651,274633,264788,253595,234405,243765
3,전국,2세,총인구수[명],439207,442943,411225,362900,331606,306120,277529,266619,254654,235297
4,전국,3세,총인구수[명],440530,439700,443586,412018,363250,332157,307975,279134,267456,255233
5,전국,4세,총인구수[명],489140,440872,440123,444102,412429,363613,333048,309282,279924,267994
6,서울,0세,총인구수[명],70798,61253,54719,51145,45165,43410,40742,37771,39588,43645
7,서울,1세,총인구수[명],76955,70532,60805,54779,50482,44904,43950,41087,37696,39703


어린이집 현황: 19행 × 11열


,0,1,2,3,4,5,6,7,8,9,10
0,시도별,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
1,전국,41084,40238,39171,37371,35352,33246,30923,28954,27387,26064
2,서울특별시,6368,6226,6008,5698,5370,5049,4712,4431,4212,4010
3,부산광역시,1937,1920,1891,1848,1778,1668,1547,1447,1354,1278
4,대구광역시,1483,1464,1405,1322,1268,1187,1139,1079,1035,992
5,인천광역시,2231,2186,2141,2049,1943,1803,1697,1652,1613,1556
6,광주광역시,1238,1240,1195,1122,1072,1002,940,878,828,785
7,대전광역시,1584,1505,1406,1288,1185,1101,1013,916,864,826



[numerator_active] 전국_어린이집_정현원_현황_20260724113247.csv.xlsx
전국_어린이집_정현원_현황_20260724113247: 19행 × 12열


,0,1,2,3,4,5,6,7,8,9,10,11
0,시도별,항목,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
1,전국,정원,1767224,1756603,1732324,1686873,1628260,1557496,1476686,1401440,1340087,1285032
2,서울특별시,정원,270231,268100,263157,254538,245863,235682,223628,211248,202722,195742
3,부산광역시,정원,89658,89040,88222,86553,84008,80248,76146,71570,66235,62266
4,대구광역시,정원,75255,74696,72500,69390,66727,62125,59434,56462,53946,52133
5,인천광역시,정원,94314,93938,93888,91615,88002,84877,80896,78883,76831,75036
6,광주광역시,정원,62528,63161,61121,57756,55020,49556,46531,43399,41021,38868
7,대전광역시,정원,54514,53688,51025,47923,45160,42171,39145,36356,34983,34111


### 비교용 어린이집 수 자료

기존 가공값이 어린이집 수를 사용했는지 확인한다. 이 자료도 별도 파일이 아니라 `finance_team_value`의 "어린이집 현황" 시트에 이미 들어있어, 여기서 바로 읽는다.

In [15]:
# cell 11

facility_count_raw = pd.read_excel(
    childcare_paths["finance_team_value"], sheet_name="어린이집 현황"
)

print(f"크기: {facility_count_raw.shape[0]}행 × {facility_count_raw.shape[1]}열")
print(f"열 이름: {facility_count_raw.columns.tolist()}")

display(facility_count_raw.head(8))

크기: 18행 × 11열
열 이름: ['시도별', 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


,시도별,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,전국,41084,40238,39171,37371,35352,33246,30923,28954,27387,26064
1,서울특별시,6368,6226,6008,5698,5370,5049,4712,4431,4212,4010
2,부산광역시,1937,1920,1891,1848,1778,1668,1547,1447,1354,1278
3,대구광역시,1483,1464,1405,1322,1268,1187,1139,1079,1035,992
4,인천광역시,2231,2186,2141,2049,1943,1803,1697,1652,1613,1556
5,광주광역시,1238,1240,1195,1122,1072,1002,940,878,828,785
6,대전광역시,1584,1505,1406,1288,1185,1101,1013,916,864,826
7,울산광역시,895,881,868,842,790,720,656,612,569,535


## 5. 정원·시설 수·0–4세 인구 결합

지역명을 `서울특별시 → 서울`처럼 통일하고 세 자료를 `지역 × 연도` 긴 형식으로 변환한다. 0–4세 인구는 별도 파일이 아니라 `finance_team_value`의 "0-4세 인구" 시트에서 읽어 `총인구수[명]`만 합산한다. 이후 세 자료를 일대일 결합하고, 18개 지역 × 10개 연도인 180행과 결측값 없음까지 검증한다. 결과 변수는 `childcare_comparison`이다.

In [ ]:
# cell 12
# region_order/region_map은 위쪽 "공통 설정" 셀에서 이미 정의됨(이 노트북 전체 공유)

years = [str(y) for y in range(2016, 2026)]


def wide_to_long(df, region_col, value_col):
    df = df.copy()
    df.columns = df.columns.map(lambda x: str(x).strip())
    df["지역"] = df[region_col].astype(str).str.strip().map(region_map)

    if df["지역"].isna().any():
        raise ValueError(f"{value_col}: 미등록 지역명이 있습니다.")
    if set(df["지역"]) != set(region_order) or df["지역"].duplicated().any():
        raise ValueError(f"{value_col}: 지역 누락 또는 중복이 있습니다.")

    result = df[["지역", *years]].melt(id_vars="지역", var_name="연도", value_name=value_col)
    result["연도"] = result["연도"].astype(int)
    result[value_col] = pd.to_numeric(result[value_col], errors="raise")
    return result


capacity_raw = pd.read_excel(childcare_paths["numerator_active"])

population_raw = pd.read_excel(childcare_paths["finance_team_value"], sheet_name="0-4세 인구")
population_raw.columns = population_raw.columns.map(lambda x: str(x).strip())
population_raw = population_raw.rename(columns={f"{y} 년": y for y in years})

for col in ["지역", "연령별", "항목"]:
    population_raw[col] = population_raw[col].astype(str).str.strip()

population_selected = population_raw[
    population_raw["지역"].isin(region_map)
    & population_raw["연령별"].isin([f"{age}세" for age in range(5)])
    & population_raw["항목"].eq("총인구수[명]")
].copy()

population_selected["지역"] = population_selected["지역"].map(region_map)

age_counts = population_selected.groupby("지역").size().reindex(region_order, fill_value=0)
if not age_counts.eq(5).all():
    raise ValueError(f"지역별 0–4세 자료 오류:\n{age_counts[age_counts.ne(5)]}")

population_selected[years] = population_selected[years].apply(pd.to_numeric, errors="raise")
population_wide = population_selected.groupby("지역", as_index=False)[years].sum()

childcare_comparison = (
    wide_to_long(capacity_raw, "시도별", "어린이집_정원")
    .merge(
        wide_to_long(facility_count_raw, "시도별", "어린이집_수"),
        on=["지역", "연도"],
        validate="one_to_one",
    )
    .merge(
        wide_to_long(population_wide, "지역", "0_4세_인구"),
        on=["지역", "연도"],
        validate="one_to_one",
    )
)

order_map = {r: i for i, r in enumerate(region_order)}
childcare_comparison["_순서"] = childcare_comparison["지역"].map(order_map)
childcare_comparison = (
    childcare_comparison.sort_values(["_순서", "연도"]).drop(columns="_순서").reset_index(drop=True)
)

if len(childcare_comparison) != 180 or childcare_comparison.isna().any().any():
    raise ValueError("결합 결과에 행 수 오류 또는 결측값이 있습니다.")

print(f"결합 결과: {childcare_comparison.shape}")
display(childcare_comparison.head(12))

## 6. 기존 산식 오류 검증

두 산식을 같은 지역·연도 자료로 계산한다.

- 정원 기준: `어린이집 정원 ÷ 0–4세 인구 × 100`
- 개수 기준: `어린이집 수 ÷ 0–4세 인구 × 100`

기존 가공값과 개수 기준 값의 절대오차를 계산한다. `기존값 최대 오차: 0.0`은 180개 지역·연도 값 모두가 개수 기준 산식과 정확히 일치한다는 뜻이며, 기존 산식 오류가 확정된다.

In [17]:
# cell 13

childcare_comparison["정원_기준_보급률"] = (
    childcare_comparison["어린이집_정원"] / childcare_comparison["0_4세_인구"] * 100
)
childcare_comparison["개수_기준_보급률"] = (
    childcare_comparison["어린이집_수"] / childcare_comparison["0_4세_인구"] * 100
)

existing_raw = pd.read_excel(
    childcare_paths["finance_team_value"], sheet_name="2-1.1. 보육시설 보급률"
)
existing_rate = wide_to_long(existing_raw, "지역", "기존_보급률")

childcare_comparison = childcare_comparison.merge(
    existing_rate, on=["지역", "연도"], validate="one_to_one"
)
childcare_comparison["기존값_오차"] = (
    childcare_comparison["기존_보급률"] - childcare_comparison["개수_기준_보급률"]
).abs()

print("기존값 최대 오차:", childcare_comparison["기존값_오차"].max())
display(childcare_comparison.head(12).round(6))

기존값 최대 오차: 0.0


,지역,연도,어린이집_정원,어린이집_수,0_4세_인구,정원_기준_보급률,개수_기준_보급률,기존_보급률,기존값_오차
0,전국,2016,1767224,41084,2204271,80.172719,1.863836,1.863836,0.0
1,전국,2017,1756603,40238,2079115,84.488015,1.935343,1.935343,0.0
2,전국,2018,1732324,39171,1974244,87.746196,1.984101,1.984101,0.0
3,전국,2019,1686873,37371,1845122,91.423386,2.025395,2.025395,0.0
4,전국,2020,1628260,35352,1677023,97.092288,2.108021,2.108021,0.0
5,전국,2021,1557496,33246,1530469,101.765929,2.172275,2.172275,0.0
6,전국,2022,1476686,30923,1427590,103.439083,2.166098,2.166098,0.0
7,전국,2023,1401440,28954,1334588,105.009186,2.169508,2.169508,0.0
8,전국,2024,1340087,27387,1271776,105.371308,2.153445,2.153445,0.0
9,전국,2025,1285032,26064,1254501,102.433717,2.077639,2.077639,0.0


## 7. 수정값 표 생성과 인계 상태

`정원_기준_보급률`을 기존 가공 파일과 같은 `지역 × 연도` 형태로 변환해 `childcare_corrected`에 저장한다. 정상 결과는 18행 × 11열이며 결측값이 없어야 한다.

**현재 완료:** 보육시설 보급률의 원자료 연결, 기존 오류 입증, 2016–2025년 수정값 생성.

**후속 작업:** 수정값을 실제 가공 파일·통합본에 반영하고 일치 여부를 재검증한 뒤, 나머지 구조환경지표의 산식 검증을 이어간다.

In [18]:
# cell 14: 수정된 보육시설 보급률 표 생성

childcare_corrected = (
    childcare_comparison.pivot(index="지역", columns="연도", values="정원_기준_보급률")
    .reindex(index=region_order, columns=range(2016, 2026))
    .reset_index()
)

if childcare_corrected.shape != (18, 11) or childcare_corrected.isna().any().any():
    raise ValueError("수정 결과의 지역·연도 또는 결측값을 확인하세요.")

display(childcare_corrected.round(6))

연도,지역,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,전국,80.172719,84.488015,87.746196,91.423386,97.092288,101.765929,103.439083,105.009186,105.371308,102.433717
1,서울,71.760207,76.539425,81.103646,84.620627,90.898436,96.378477,98.246631,99.170947,99.913749,96.602131
2,부산,67.900094,71.934077,75.343530,80.169132,86.273543,91.421541,93.692785,94.970807,93.110380,88.705587
3,대구,75.863425,79.655342,81.366508,83.797264,89.928571,93.622376,97.079481,98.046434,98.308853,95.607761
4,인천,72.337782,77.077966,81.379203,85.178092,91.364203,95.631746,94.561012,95.323432,94.397415,90.429878
5,광주,95.659757,103.081292,105.433751,107.090410,112.815255,109.276941,109.487282,111.519683,113.663065,112.273607
6,대전,79.918489,84.948023,86.960597,89.703130,94.706820,97.009501,94.069161,91.505374,91.449260,89.494949
7,울산,69.930689,74.560186,78.919881,85.237655,90.700375,94.685686,96.302139,98.812328,98.323008,95.434192
8,세종,73.850035,76.240508,84.151680,87.735249,93.450074,97.494749,99.077199,108.238983,112.987013,109.152997
9,경기,78.161356,81.743124,84.387387,88.428539,93.587910,97.861527,99.529469,102.370861,102.568124,99.795011


## 8. 청년고용률(1-1.1) 검증

`1-1.1. 청년고용률.xlsx`는 시트 4개(`1-1.청년고용률`, `청년고용률 계산`, `20-34세 인구`, `20-34세 취업자 수`)를 갖는다. 4단계로 대조한다: (1) 계산 시트 내부 재계산, (2) 계산 시트 vs 결과 시트, (3) 원자료 개별연령 인구 합산 vs 계산 시트 인구, (4) 원자료 연령대별 취업자 합산 vs 계산 시트 취업자.

In [19]:
# 8-1. 파일 로드

youth = next(item for item in indicators if item["id"] == "youth_employment_rate")
youth_path = find_matches(paths_by_name, youth["finance_team_value"]["file_name"])[0]
print(f"finance_team_value: {youth_path.relative_to(repo_root)}")

years10 = list(range(2016, 2026))

youth_calc = pd.read_excel(youth_path, sheet_name="청년고용률 계산", header=0)
youth_calc.columns = [str(c).strip() for c in youth_calc.columns]
youth_calc = youth_calc.rename(columns={f"{y} 년": y for y in years10})
youth_calc["행정구역(시군구)별"] = youth_calc["행정구역(시군구)별"].astype(str).str.strip()

youth_pop_calc = (
    youth_calc[youth_calc.iloc[:, 1] == "인구수(천명)"]
    .set_index("행정구역(시군구)별")[years10]
    .apply(pd.to_numeric, errors="coerce")
)
youth_emp_calc = (
    youth_calc[youth_calc.iloc[:, 1] == "취업자수(천명)"]
    .set_index("행정구역(시군구)별")[years10]
    .apply(pd.to_numeric, errors="coerce")
)
youth_rate_calc = (
    youth_calc[youth_calc.iloc[:, 1] == "청년고용률(%)"]
    .set_index("행정구역(시군구)별")[years10]
    .apply(pd.to_numeric, errors="coerce")
)

print(f"계산 시트 지역 수: {len(youth_pop_calc)}")

finance_team_value: data/raw/지표별_측정값/1-1.1. 청년고용률.xlsx
계산 시트 지역 수: 18


In [20]:
# 8-2. 내부 재계산 및 결과 시트 대조

youth_recompute = youth_emp_calc / youth_pop_calc * 100
youth_diff_internal = (youth_recompute - youth_rate_calc).abs()
print("계산 시트 내부 재계산 오차 최대:", youth_diff_internal.max().max())

youth_result = pd.read_excel(youth_path, sheet_name="1-1.청년고용률")
youth_result.columns = [c if isinstance(c, int) else str(c).strip() for c in youth_result.columns]
youth_result["지역"] = youth_result["지역"].astype(str).str.strip()
youth_result_wide = youth_result.set_index("지역")[years10].apply(pd.to_numeric, errors="coerce")

youth_diff_result = (
    youth_rate_calc.reindex(region_order) - youth_result_wide.reindex(region_order)
).abs()
print("계산 시트 vs 결과 시트 오차 최대:", youth_diff_result.max().max())

계산 시트 내부 재계산 오차 최대: 0.0
계산 시트 vs 결과 시트 오차 최대: 0.0


In [ ]:
# 8-3. 원자료(개별 연령·연령대) 재집계 대조
# 지역명 매핑은 8-2까지 정의된 공통 region_map(56457f64 셀)을 그대로 재사용한다.

youth_pop_raw = pd.read_excel(youth_path, sheet_name="20-34세 인구")
youth_pop_raw.columns = [str(c).strip() for c in youth_pop_raw.columns]
youth_pop_raw = youth_pop_raw.rename(columns={f"{y} 년": y for y in years10})
individual_ages = [f"{age}세" for age in range(20, 35)]
youth_pop_ind = youth_pop_raw[youth_pop_raw["연령별"].isin(individual_ages)].copy()
youth_pop_ind["지역"] = youth_pop_ind["행정구역(시군구)별"].map(region_map)
youth_pop_sum = youth_pop_ind.dropna(subset=["지역"]).groupby("지역")[years10].sum() / 1000

youth_diff_pop = (youth_pop_sum.reindex(region_order) - youth_pop_calc.reindex(region_order)).abs()
print("원자료 개별연령 합산 vs 계산 시트 인구 오차 최대:", youth_diff_pop.max().max())

youth_emp_raw = pd.read_excel(youth_path, sheet_name="20-34세 취업자 수")
youth_emp_raw.columns = [c if isinstance(c, int) else str(c).strip() for c in youth_emp_raw.columns]
youth_emp_raw["시도별(1)"] = youth_emp_raw["시도별(1)"].astype(str).str.strip()
bands = ["20 - 24세", "25 - 29세", "30 - 34세"]
youth_emp_bands = youth_emp_raw[youth_emp_raw["연령계층별(2)"].isin(bands)].copy()
youth_emp_bands["지역"] = youth_emp_bands["시도별(1)"].map(region_map)
youth_emp_sum = (
    youth_emp_bands.dropna(subset=["지역"])
    .groupby("지역")[years10]
    .apply(lambda g: g.apply(pd.to_numeric, errors="coerce").sum())
)

youth_diff_emp = (youth_emp_sum.reindex(region_order) - youth_emp_calc.reindex(region_order)).abs()
print("원자료 연령대별 취업자 합산 vs 계산 시트 취업자 오차 최대:", youth_diff_emp.max().max())

## 9. 방과후 돌봄시설 보급도(2-1.2) 검증

official_formula: (다함께돌봄센터 수 + 지역아동센터 수) ÷ 6–12세 주민등록인구 × 1,000. 5단계로 대조한다: 내부 재계산, 결과 시트 대조, 원자료 인구 개별연령 합산, 원자료 다함께돌봄센터·지역아동센터 원본 대조.

In [22]:
# 9-1. 파일 로드 및 내부 재계산

after_school = next(item for item in indicators if item["id"] == "after_school_care_supply")
after_school_path = find_matches(paths_by_name, after_school["finance_team_value"]["file_name"])[0]
print(f"finance_team_value: {after_school_path.relative_to(repo_root)}")

as_calc = pd.read_excel(after_school_path, sheet_name="방과후 돌봄시설 보급도 계산", header=0)
as_calc.columns = [c if isinstance(c, int) else str(c).strip() for c in as_calc.columns]
as_calc["지역"] = as_calc["지역"].astype(str).str.strip()

as_center1 = as_calc[as_calc["항목"] == "다함께돌봄센터"].set_index("지역")[years10]
as_center2 = as_calc[as_calc["항목"] == "지역아동센터"].set_index("지역")[years10]
as_pop = as_calc[as_calc["항목"] == "6-12세"].set_index("지역")[years10]
as_rate_calc = as_calc[as_calc["항목"] == "방과후 돌봄시설 보급도(인구천명당)"].set_index("지역")[
    years10
]

as_recompute = (as_center1 + as_center2) / as_pop * 1000
as_diff_internal = (as_recompute - as_rate_calc).abs()
print("내부 재계산 오차 최대:", as_diff_internal.max().max())

finance_team_value: data/raw/지표별_측정값/2-1.2. 방과후 돌봄시설 보급도.xlsx
내부 재계산 오차 최대: 0.0


In [23]:
# 9-2. 결과 시트 및 원자료 대조

as_result = pd.read_excel(after_school_path, sheet_name="2-1.2.방과후 돌봄시설 보급도")
as_result.columns = [c if isinstance(c, int) else str(c).strip() for c in as_result.columns]
as_result["지역"] = as_result["지역"].astype(str).str.strip()
as_result_wide = as_result.set_index("지역")[years10]
as_diff_result = (as_rate_calc.reindex(region_order) - as_result_wide.reindex(region_order)).abs()
print("계산 시트 vs 결과 시트 오차 최대:", as_diff_result.max().max())

as_pop_raw = pd.read_excel(after_school_path, sheet_name="6-12세 인구수")
as_pop_raw.columns = [str(c).strip() for c in as_pop_raw.columns]
as_pop_raw = as_pop_raw.rename(columns={f"{y} 년": y for y in years10})
as_pop_raw["지역"] = as_pop_raw["지역"].astype(str).str.strip()
individual_ages_6_12 = [f"{age}세" for age in range(6, 13)]
as_pop_sum = (
    as_pop_raw[as_pop_raw["연령별"].isin(individual_ages_6_12)].groupby("지역")[years10].sum()
)
as_diff_pop = (as_pop_sum.reindex(region_order) - as_pop.reindex(region_order)).abs()
print("원자료 개별연령 합산 vs 계산 시트 인구 오차 최대:", as_diff_pop.max().max())

as_fac_raw = pd.read_excel(after_school_path, sheet_name="방과후 돌봄시설 수")
as_fac_raw.columns = [c if isinstance(c, int) else str(c).strip() for c in as_fac_raw.columns]
as_fac_raw["지역"] = as_fac_raw["지역"].astype(str).str.strip()
years9 = list(range(2016, 2025))
as_center1_raw = as_fac_raw[as_fac_raw["항목"] == "다함께돌봄센터"].set_index("지역")[years9]
as_center2_raw = as_fac_raw[as_fac_raw["항목"] == "지역아동센터"].set_index("지역")[years9]
as_diff_c1 = (as_center1_raw.reindex(region_order) - as_center1.reindex(region_order)[years9]).abs()
as_diff_c2 = (as_center2_raw.reindex(region_order) - as_center2.reindex(region_order)[years9]).abs()
print("원자료 다함께돌봄센터 오차 최대:", as_diff_c1.max().max())
print("원자료 지역아동센터 오차 최대:", as_diff_c2.max().max())

계산 시트 vs 결과 시트 오차 최대: 0.0
원자료 개별연령 합산 vs 계산 시트 인구 오차 최대: 0.0
원자료 다함께돌봄센터 오차 최대: 0.0
원자료 지역아동센터 오차 최대: 0.0


## 10. (대체)분만실 병상수 보급도(3-1.1) 검증

project_formula: 연도별 분기 평균 분만실 병상 수 ÷ 해당 연도 전체 주민등록인구 × 100,000. 원자료가 분기 단위(2018.3/4~2025.4/4)라 연도별 평균으로 환산하는 과정을 재현해서 대조한다.

In [ ]:
# 10-1. 파일 로드, 내부 재계산, 결과 시트 대조

delivery = next(item for item in indicators if item["id"] == "delivery_bed_supply")
delivery_path = find_matches(paths_by_name, delivery["finance_team_value"]["file_name"])[0]
print(f"finance_team_value: {delivery_path.relative_to(repo_root)}")

delivery_calc = pd.read_excel(delivery_path, sheet_name="분만실병상수보급도 계산", header=0)
delivery_calc.columns = [str(c).strip() for c in delivery_calc.columns]
delivery_calc = delivery_calc.rename(columns={f"{y} 년": y for y in years10})
delivery_calc["지역"] = delivery_calc["지역"].astype(str).str.strip()

delivery_pop = delivery_calc[delivery_calc["항목"] == "총인구수[명]"].set_index("지역")[years10]
delivery_bed_avg = delivery_calc[
    delivery_calc["항목"] == "분기별 분만실 병상수 평균(개)"
].set_index("지역")[years10]
delivery_rate_calc = delivery_calc[
    delivery_calc["항목"] == "분만실 병상수 보급도(인구 십만명당)"
].set_index("지역")[years10]

delivery_recompute = delivery_bed_avg / delivery_pop * 100000
delivery_diff_internal = (delivery_recompute - delivery_rate_calc).abs()
print("내부 재계산 오차 최대:", delivery_diff_internal.max().max())

delivery_result = pd.read_excel(delivery_path, sheet_name="3-1.1. 분만실 병상수 보급도")
delivery_result.columns = [
    c if isinstance(c, int) else str(c).strip() for c in delivery_result.columns
]
delivery_result["지역"] = delivery_result["지역"].astype(str).str.strip().map(region_map)
delivery_result_wide = delivery_result.set_index("지역")[years10]
delivery_diff_result = (
    delivery_rate_calc.reindex(region_order) - delivery_result_wide.reindex(region_order)
).abs()
print("계산 시트 vs 결과 시트 오차 최대:", delivery_diff_result.max().max())

In [ ]:
# 10-2. 원자료 분기별 병상수를 직접 연도 평균으로 재계산해서 대조

delivery_raw = pd.read_excel(delivery_path, sheet_name="분만실 수")
delivery_raw.columns = [str(c).strip() for c in delivery_raw.columns]
delivery_raw["지역"] = delivery_raw["지역"].astype(str).str.strip().map(region_map)

delivery_raw_counts = delivery_raw[delivery_raw["항목"] == "분기별 분만실 병상수(개)"].set_index(
    "지역"
)
quarter_cols = [c for c in delivery_raw_counts.columns if "." in str(c)]

delivery_avg_by_year = {}
for year in years10:
    cols = [c for c in quarter_cols if c.startswith(f"{year}.")]
    if cols:
        delivery_avg_by_year[year] = delivery_raw_counts[cols].mean(axis=1)
delivery_avg_df = pd.DataFrame(delivery_avg_by_year)

delivery_diff_raw = (
    delivery_avg_df.reindex(region_order)
    - delivery_bed_avg.reindex(region_order)[delivery_avg_df.columns]
).abs()
print("원자료 분기평균 재계산 vs 계산 시트 오차 최대:", delivery_diff_raw.max().max())

## 11. (대체)소아청소년과 전문인력 보급도(3-1.2) 검증

3-1.1과 동일한 구조(분기별 전문의 수 평균 ÷ 인구 × 100,000). 내부 재계산과 결과 시트 대조만 확인한다.

In [ ]:
# 11-1. 파일 로드, 내부 재계산, 결과 시트 대조

pediatric = next(item for item in indicators if item["id"] == "pediatric_specialist_supply")
pediatric_path = find_matches(paths_by_name, pediatric["finance_team_value"]["file_name"])[0]
print(f"finance_team_value: {pediatric_path.relative_to(repo_root)}")

pediatric_calc = pd.read_excel(
    pediatric_path, sheet_name="소아청소년과 전문인력 보급도 계산", header=0
)
pediatric_calc.columns = [str(c).strip() for c in pediatric_calc.columns]
pediatric_calc = pediatric_calc.rename(columns={f"{y} 년": y for y in years10})
pediatric_calc["지역"] = pediatric_calc["지역"].astype(str).str.strip()

pediatric_pop = pediatric_calc[pediatric_calc["항목"] == "총인구수[명]"].set_index("지역")[years10]
pediatric_avg = pediatric_calc[pediatric_calc["항목"].str.contains("평균", na=False)].set_index(
    "지역"
)[years10]
pediatric_rate_calc = pediatric_calc[
    pediatric_calc["항목"].str.contains("보급도", na=False)
].set_index("지역")[years10]

pediatric_recompute = pediatric_avg / pediatric_pop * 100000
pediatric_diff_internal = (pediatric_recompute - pediatric_rate_calc).abs()
print("내부 재계산 오차 최대:", pediatric_diff_internal.max().max())

pediatric_result = pd.read_excel(
    pediatric_path, sheet_name="3-1.2.(대체)소아청소년과 전문인력 보급도"
)
pediatric_result.columns = [
    c if isinstance(c, int) else str(c).strip() for c in pediatric_result.columns
]
pediatric_result["지역"] = pediatric_result["지역"].astype(str).str.strip().map(region_map)
pediatric_result_wide = pediatric_result.set_index("지역")[years10]
pediatric_diff_result = (
    pediatric_rate_calc.reindex(region_order) - pediatric_result_wide.reindex(region_order)
).abs()
print("계산 시트 vs 결과 시트 오차 최대:", pediatric_diff_result.max().max())

## 12. 산후조리원 보급도(3-2.1) 검증

official_formula: 산후조리원 수 ÷ 전체 주민등록인구 × 100,000. 2023~2025년만 존재. 원자료 "산후조리원 현황(통합)"의 개별 시설 목록에서 지역·연도별로 직접 개수를 세어 계산 시트와 대조한다.

In [27]:
# 12-1. 파일 로드, 내부 재계산, 결과 시트 대조

years3 = [2023, 2024, 2025]
postpartum_supply = next(item for item in indicators if item["id"] == "postpartum_center_supply")
postpartum_supply_path = find_matches(
    paths_by_name, postpartum_supply["finance_team_value"]["file_name"]
)[0]
print(f"finance_team_value: {postpartum_supply_path.relative_to(repo_root)}")

pp_calc = pd.read_excel(postpartum_supply_path, sheet_name="산후조리원 보급도 계산", header=0)
pp_calc.columns = [str(c).strip() for c in pp_calc.columns]
pp_calc = pp_calc.rename(columns={f"{y} 년": y for y in years10})
pp_calc["지역"] = pp_calc["지역"].astype(str).str.strip()

pp_pop = pp_calc[pp_calc["항목"] == "총인구수[명]"].set_index("지역")[years3]
pp_count = pp_calc[pp_calc["항목"] == "산후조리원수(개)"].set_index("지역")[years3]
pp_rate_calc = pp_calc[pp_calc["항목"].str.contains("보급도", na=False)].set_index("지역")[years3]

pp_recompute = pp_count / pp_pop * 100000
pp_diff_internal = (pp_recompute - pp_rate_calc).abs()
print("내부 재계산 오차 최대:", pp_diff_internal.max().max())

pp_result = pd.read_excel(postpartum_supply_path, sheet_name="3-2.1. 산후조리원 보급도")
pp_result.columns = [c if isinstance(c, int) else str(c).strip() for c in pp_result.columns]
pp_result["지역"] = pp_result["지역"].astype(str).str.strip()
pp_result_wide = pp_result.set_index("지역")[years3]
pp_diff_result = (pp_rate_calc.reindex(region_order) - pp_result_wide.reindex(region_order)).abs()
print("계산 시트 vs 결과 시트 오차 최대:", pp_diff_result.max().max())

finance_team_value: data/raw/지표별_측정값/3-2.1. 산후조리원 보급도.xlsx
내부 재계산 오차 최대: 0.0
계산 시트 vs 결과 시트 오차 최대: 0.0


In [28]:
# 12-2. 원자료 개별 시설 목록에서 직접 개수 세어 대조

pp_listing = pd.read_excel(postpartum_supply_path, sheet_name="산후조리원 현황(통합)", header=4)
pp_listing_valid = pp_listing[["연도", "지역", "산후조리원"]].dropna(subset=["산후조리원"])

pp_counts_direct = pp_listing_valid.groupby(["연도", "지역"]).size().unstack("지역")
pp_counts_direct["전국"] = pp_counts_direct.sum(axis=1)
pp_counts_direct = pp_counts_direct.reindex(columns=region_order)

pp_diff_count = (pp_counts_direct.T[years3] - pp_count.reindex(region_order)[years3]).abs()
print("원자료 개별 시설 직접 카운트 vs 계산 시트 개수 오차 최대:", pp_diff_count.max().max())

원자료 개별 시설 직접 카운트 vs 계산 시트 개수 오차 최대: 0.0


## 13. 산후조리원 이용 요금(3-2.2) 검증 — 오류 확정(재발)

official_formula: 지자체별 조리원의 14일 기준 최고 **일반실** 요금 평균. **일반실이 없으면 특실 요금을 적용한다.**

이 지표는 지준구 님이 카톡에서 지적한 이슈(일반실 없는 시설에 특실요금 대체 미적용)와 동일한 문제이고, 최연호 님이 "수정 완료"라고 답했던 항목이다. 아래에서 "일반실만(결측 제외)"과 "일반실 없으면 특실 대체(공식 산식)" 두 방식으로 각각 재계산해서 재정팀 선언값과 대조한다.

In [29]:
# 13-1. 파일 로드 및 두 가지 방식으로 재계산

postpartum_fee = next(item for item in indicators if item["id"] == "postpartum_center_fee")
postpartum_fee_path = find_matches(
    paths_by_name, postpartum_fee["finance_team_value"]["file_name"]
)[0]
print(f"finance_team_value: {postpartum_fee_path.relative_to(repo_root)}")

fee_listing = pd.read_excel(postpartum_fee_path, sheet_name="산후조리원 현황(통합)", header=4)
fee_listing = (
    fee_listing[["연도", "지역", "산후조리원", "일반실", "특실"]]
    .dropna(subset=["산후조리원"])
    .copy()
)

missing_count = fee_listing["일반실"].isna().sum()
print(f"일반실 결측 시설: {missing_count}건 / 전체 {len(fee_listing)}건")

# 방식 A: 일반실만(결측 시설 제외) - 재정팀이 실제로 쓴 것으로 의심되는 방식
fee_avg_only = fee_listing.groupby(["연도", "지역"])["일반실"].mean().unstack("지역")
fee_avg_only["전국"] = fee_listing.groupby("연도")["일반실"].mean()
fee_avg_only = fee_avg_only.reindex(columns=region_order)

# 방식 B: 일반실 없으면 특실로 대체(공식 산식)
fee_listing["요금_대체"] = fee_listing["일반실"].fillna(fee_listing["특실"])
fee_avg_fallback = fee_listing.groupby(["연도", "지역"])["요금_대체"].mean().unstack("지역")
fee_avg_fallback["전국"] = fee_listing.groupby("연도")["요금_대체"].mean()
fee_avg_fallback = fee_avg_fallback.reindex(columns=region_order)

finance_team_value: data/raw/지표별_측정값/3-2.2. 산후조리원 이용 요금.xlsx
일반실 결측 시설: 24건 / 전체 1367건


In [30]:
# 13-2. 재정팀 선언값과 두 방식 대조 - 오류 확정

fee_result = pd.read_excel(postpartum_fee_path, sheet_name="3-2.1. 산후조리원 보급도")
fee_result.columns = [c if isinstance(c, int) else str(c).strip() for c in fee_result.columns]
fee_result["지역"] = fee_result["지역"].astype(str).str.strip()
fee_result_wide = fee_result.set_index("지역")[years3]

fee_diff_only = (fee_avg_only.T[years3] - fee_result_wide.reindex(region_order)).abs()
fee_diff_fallback = (fee_avg_fallback.T[years3] - fee_result_wide.reindex(region_order)).abs()

print("방식 A(일반실만, 결측 제외) vs 재정팀 선언값 최대 오차:", fee_diff_only.max().max())
print(
    "방식 B(일반실 없으면 특실 대체, 공식 산식) vs 재정팀 선언값 최대 오차:",
    fee_diff_fallback.max().max(),
)
print(
    "\n결론: 방식 A가 재정팀 선언값과 사실상 일치(부동소수점 오차 수준)하고, "
    "공식 산식대로인 방식 B는 최대",
    round(fee_diff_fallback.max().max(), 2),
    "만원 차이 -> 특실 대체 로직이 반영돼 있지 않다.",
)

방식 A(일반실만, 결측 제외) vs 재정팀 선언값 최대 오차: 5.684341886080802e-14
방식 B(일반실 없으면 특실 대체, 공식 산식) vs 재정팀 선언값 최대 오차: 22.690909090909088

결론: 방식 A가 재정팀 선언값과 사실상 일치(부동소수점 오차 수준)하고, 공식 산식대로인 방식 B는 최대 22.69 만원 차이 -> 특실 대체 로직이 반영돼 있지 않다.


## 14. 어린이 교통사고 발생률(3-3.1) 검증

official_formula: 12세 이하 어린이 교통사고 발생건수 ÷ 12세 이하 주민등록인구 × 100,000.

In [31]:
# 14-1. 파일 로드, 내부 재계산, 결과 시트 대조

child_accident = next(item for item in indicators if item["id"] == "child_traffic_accident_rate")
child_accident_path = find_matches(
    paths_by_name, child_accident["finance_team_value"]["file_name"]
)[0]
print(f"finance_team_value: {child_accident_path.relative_to(repo_root)}")

ca_calc = pd.read_excel(child_accident_path, sheet_name="어린이 교통사고 발생률 계산", header=0)
ca_calc.columns = [c if isinstance(c, int) else str(c).strip() for c in ca_calc.columns]
ca_calc["지역"] = ca_calc["지역"].astype(str).str.strip()

ca_pop = ca_calc[ca_calc["항목"] == "0-12세 인구수(명)"].set_index("지역")[years10]
ca_acc = ca_calc[ca_calc["항목"] == "어린이 교통사고 발생건수"].set_index("지역")[years10]
ca_rate_calc = ca_calc[ca_calc["항목"] == "어린이 교통사고 발생률(인구 십만명 당)"].set_index(
    "지역"
)[years10]

ca_recompute = ca_acc / ca_pop * 100000
ca_diff_internal = (ca_recompute - ca_rate_calc).abs()
print("내부 재계산 오차 최대:", ca_diff_internal.max().max())

ca_result = pd.read_excel(child_accident_path, sheet_name="3-3.1. 어린이 교통사고 발생률")
ca_result.columns = [c if isinstance(c, int) else str(c).strip() for c in ca_result.columns]
ca_result["지역"] = ca_result["지역"].astype(str).str.strip()
ca_result_wide = ca_result.set_index("지역")[years10]
ca_diff_result = (ca_rate_calc.reindex(region_order) - ca_result_wide.reindex(region_order)).abs()
print("계산 시트 vs 결과 시트 오차 최대:", ca_diff_result.max().max())

finance_team_value: data/raw/지표별_측정값/3-3.1. 어린이 교통사고 발생률.xlsx
내부 재계산 오차 최대: 0.0
계산 시트 vs 결과 시트 오차 최대: 0.0


In [ ]:
# 14-2. 원자료(개별 연령 인구·발생건수) 재집계 대조

ca_pop_raw = pd.read_excel(child_accident_path, sheet_name="12세 이하 인구수")
ca_pop_raw.columns = [str(c).strip() for c in ca_pop_raw.columns]
ca_pop_raw = ca_pop_raw.rename(columns={f"{y} 년": y for y in years10})
individual_ages_0_12 = [f"{age}세" for age in range(0, 13)]
ca_pop_ind = ca_pop_raw[ca_pop_raw["연령별"].isin(individual_ages_0_12)].copy()
ca_pop_ind["지역"] = ca_pop_ind["행정구역(시군구)별"].map(region_map)
ca_pop_sum = ca_pop_ind.dropna(subset=["지역"]).groupby("지역")[years10].sum()
ca_diff_pop = (ca_pop_sum.reindex(region_order) - ca_pop.reindex(region_order)).abs()
print("원자료 개별연령 합산 vs 계산 시트 인구 오차 최대:", ca_diff_pop.max().max())

ca_acc_raw = pd.read_excel(child_accident_path, sheet_name="어린이 교통사고 발생건수")
ca_acc_raw.columns = [
    int(c) if str(c).strip().isdigit() else str(c).strip() for c in ca_acc_raw.columns
]
ca_acc_raw["지역"] = ca_acc_raw["지역"].astype(str).str.strip()
ca_acc_raw_wide = ca_acc_raw.set_index("지역")[years10]
ca_diff_acc = (ca_acc_raw_wide.reindex(region_order) - ca_acc.reindex(region_order)).abs()
print("원자료 발생건수 vs 계산 시트 발생건수 오차 최대:", ca_diff_acc.max().max())

## 15. 사회 안전에 대한 인식(3-3.2) 검증

official_formula: 사회조사 5점 응답을 긍정 방향으로 역코딩한 가중평균(매우 안전=5 ~ 매우 안전하지 않음=1). 공표된 응답 비율 5개 항목의 합이 반올림 때문에 정확히 100이 아닌 연도가 있어, "100으로 나눔"이 아니라 "5개 항목 실제 합으로 나눔" 기준으로 대조한다.

In [33]:
# 15-1. 파일 로드 및 가중평균 재계산

social_safety = next(item for item in indicators if item["id"] == "social_safety_perception")
social_safety_path = find_matches(paths_by_name, social_safety["finance_team_value"]["file_name"])[
    0
]
print(f"finance_team_value: {social_safety_path.relative_to(repo_root)}")

ss_raw = pd.read_excel(
    social_safety_path, sheet_name="2016,2018,2020,2022,2024_사회안전에_", header=None
)
ss_years_row = ss_raw.iloc[0]
ss_cat_row = ss_raw.iloc[1]
ss_data = ss_raw.iloc[2:].reset_index(drop=True)
ss_data.columns = [
    f"{ss_years_row[i]}_{ss_cat_row[i]}" if pd.notna(ss_cat_row[i]) else ss_years_row[i]
    for i in range(len(ss_cat_row))
]
ss_data = ss_data.rename(columns={ss_data.columns[0]: "지역", ss_data.columns[1]: "항목"})

ss_scores = {
    "매우 안전": 5,
    "비교적 안전": 4,
    "보통": 3,
    "비교적 안전하지 않음": 2,
    "매우 안전하지 않음": 1,
}
ss_result = pd.read_excel(social_safety_path, sheet_name="3-3.2. 사회 안전에 대한 인식")
ss_result.columns = [c if isinstance(c, int) else str(c).strip() for c in ss_result.columns]
ss_result["지역"] = ss_result["지역"].astype(str).str.strip()

ss_max_diff = 0
for year in [2016, 2018, 2020, 2022, 2024]:
    for region in region_order:
        row = ss_data[ss_data["지역"] == region]
        if row.empty:
            continue
        row = row.iloc[0]
        catsum = sum(row[f"{year}_{cat}"] for cat in ss_scores)
        wsum = sum(row[f"{year}_{cat}"] * score for cat, score in ss_scores.items())
        computed = wsum / catsum
        declared = ss_result[ss_result["지역"] == region][year].iloc[0]
        ss_max_diff = max(ss_max_diff, abs(computed - declared))

print("응답 실제 합으로 나눈 재계산 vs 선언값 최대 오차:", ss_max_diff)

finance_team_value: data/raw/지표별_측정값/3-3.2. 사회 안전에 대한 인식.xlsx
응답 실제 합으로 나눈 재계산 vs 선언값 최대 오차: 1.3322676295501878e-15


## 16. 소득만족도(1-3.1) 검증

official_formula: 사회조사 소득만족도 5점 응답을 긍정 방향으로 역코딩한 가중평균(소득 없음 응답 제외). 원자료의 5개 응답 항목(매우만족~매우불만족)은 이미 "소득 있음" 응답자 기준 비율이라, "소득 있음" 값과 무관하게 5개 항목 합으로 나눠서 재계산한다.

In [34]:
# 16-1. 파일 로드 및 가중평균 재계산

income_satisfaction = next(item for item in indicators if item["id"] == "income_satisfaction")
income_satisfaction_path = find_matches(
    paths_by_name, income_satisfaction["finance_team_value"]["file_name"]
)[0]
print(f"finance_team_value: {income_satisfaction_path.relative_to(repo_root)}")

is_raw = pd.read_excel(
    income_satisfaction_path, sheet_name="2015,2017,2019,2021,2023,2025_소", header=None
)
is_years_row = is_raw.iloc[0]
is_cat_row = is_raw.iloc[1]
is_data = is_raw.iloc[2:].reset_index(drop=True)
is_cols = []
for i in range(len(is_cat_row)):
    is_cols.append(
        is_cat_row[i] if pd.isna(is_years_row[i]) else f"{int(is_years_row[i])}_{is_cat_row[i]}"
    )
is_data.columns = is_cols
is_data = is_data.rename(columns={is_data.columns[0]: "지역"})

is_scores = {"매우만족": 5, "약간만족": 4, "보통": 3, "약간불만족": 2, "매우불만족": 1}
is_result = pd.read_excel(income_satisfaction_path, sheet_name="1-3.1.소득만족도")
is_result.columns = [c if isinstance(c, int) else str(c).strip() for c in is_result.columns]
is_result["지역"] = is_result["지역"].astype(str).str.strip()

is_max_diff = 0
for year in [2015, 2017, 2019, 2021, 2023, 2025]:
    for region in region_order:
        row = is_data[is_data["지역"] == region]
        if row.empty:
            continue
        row = row.iloc[0]
        catsum = sum(row[f"{year}_{cat}"] for cat in is_scores)
        wsum = sum(row[f"{year}_{cat}"] * score for cat, score in is_scores.items())
        computed = wsum / catsum
        declared = is_result[is_result["지역"] == region][year].iloc[0]
        is_max_diff = max(is_max_diff, abs(computed - declared))

print("응답 실제 합으로 나눈 재계산 vs 선언값 최대 오차:", is_max_diff)

finance_team_value: data/raw/지표별_측정값/1-3.1. 소득만족도.xlsx
응답 실제 합으로 나눈 재계산 vs 선언값 최대 오차: 8.881784197001252e-16


## 17. 육아휴직 사용률(4-1.2) 검증

official_formula: 육아휴직통계의 출생아 부모 육아휴직 사용률 공표값(별도 산식 없음, 원자료의 "소계" 값을 그대로 사용).

In [35]:
# 17-1. 파일 로드 및 대조

parental_leave = next(item for item in indicators if item["id"] == "parental_leave_usage")
parental_leave_path = find_matches(
    paths_by_name, parental_leave["finance_team_value"]["file_name"]
)[0]
print(f"finance_team_value: {parental_leave_path.relative_to(repo_root)}")

years9_2024 = list(range(2016, 2025))

pl_raw = pd.read_excel(
    parental_leave_path, sheet_name="2016-2024_시도별_출생아_부모의_육아휴직_사용률_", header=0
)
pl_raw.columns = [str(c).strip() for c in pl_raw.columns]
pl_raw = pl_raw.rename(columns={"2024 p)": "2024"})
pl_raw.columns = [int(c) if str(c).isdigit() else c for c in pl_raw.columns]
pl_raw["행정구역(시도)별(1)"] = pl_raw["행정구역(시도)별(1)"].astype(str).str.strip()
pl_sub = pl_raw[pl_raw["성별(2)"] == "소계"].set_index("행정구역(시도)별(1)")[years9_2024]

pl_result = pd.read_excel(parental_leave_path, sheet_name="4-1.1. 육아휴직 사용률")
pl_result.columns = [c if isinstance(c, int) else str(c).strip() for c in pl_result.columns]
pl_result["지역"] = pl_result["지역"].astype(str).str.strip()
pl_result_wide = pl_result.set_index("지역")[years9_2024]

pl_diff = (pl_sub.reindex(region_order) - pl_result_wide.reindex(region_order)).abs()
print("원자료 소계 vs 결과 시트 오차 최대:", pl_diff.max().max())

finance_team_value: data/raw/지표별_측정값/4-1.2. 육아휴직 사용률.xlsx
원자료 소계 vs 결과 시트 오차 최대: 0.0


## 18. 가족친화 인증기업 비율(4-1.3) 검증 — 경미한 불일치 발견(원인 미상)

official_formula: 가족친화인증기업·기관 수 ÷ 전체 사업체 수 × 100. 데이터는 2018·2022·2023·2024·2025년만 존재. 인증기업 수(분자)를 원자료 "가족친화 인증기업 현황"(개별 기업 목록)에서 지역·연도별로 직접 세어 계산 시트와 대조한다. 2024년 원자료만 "지역" 컬럼이 시군구 상세 주소라 시도 단위로 재매핑한다.

In [36]:
# 18-1. 파일 로드, 내부 재계산, 사업체 수(분모) 대조

years5 = [2018, 2022, 2023, 2024, 2025]
family_friendly = next(
    item for item in indicators if item["id"] == "family_friendly_certification_rate"
)
family_friendly_path = find_matches(
    paths_by_name, family_friendly["finance_team_value"]["file_name"]
)[0]
print(f"finance_team_value: {family_friendly_path.relative_to(repo_root)}")

ff_calc = pd.read_excel(family_friendly_path, sheet_name="가족친화인증기업 비율 계산", header=0)
ff_calc.columns = [c if isinstance(c, int) else str(c).strip() for c in ff_calc.columns]
ff_calc["지역"] = ff_calc["지역"].astype(str).str.strip()

ff_denom = ff_calc[ff_calc["항목"] == "전산업/전규모 사업체수(개)"].set_index("지역")[years5]
ff_numer = ff_calc[ff_calc["항목"] == "가족친화인증기업 수(개)"].set_index("지역")[years5]
ff_rate_calc = ff_calc[ff_calc["항목"] == "가족친화인증기업 비율(%)"].set_index("지역")[years5]

ff_recompute = ff_numer / ff_denom * 100
ff_diff_internal = (ff_recompute - ff_rate_calc).abs()
print("내부 재계산 오차 최대:", ff_diff_internal.max().max())

ff_result = pd.read_excel(family_friendly_path, sheet_name="4-1.2. 가족친화 인증기업 비율")
ff_result.columns = [c if isinstance(c, int) else str(c).strip() for c in ff_result.columns]
ff_result["지역"] = ff_result["지역"].astype(str).str.strip()
ff_result_wide = ff_result.set_index("지역")[years5]
ff_diff_result = (ff_rate_calc.reindex(region_order) - ff_result_wide.reindex(region_order)).abs()
print("계산 시트 vs 결과 시트 오차 최대:", ff_diff_result.max().max())

ff_biz = pd.read_excel(family_friendly_path, sheet_name="사업체 수", header=0, skiprows=[1])
ff_biz.columns = [int(c) if isinstance(c, (int, float)) else str(c).strip() for c in ff_biz.columns]
ff_biz["시도별(17개)"] = ff_biz["시도별(17개)"].astype(str).str.strip()
ff_biz_wide = ff_biz.set_index("시도별(17개)")
ff_biz_years = [y for y in years5 if y in ff_biz_wide.columns]
ff_diff_denom = (
    ff_biz_wide[ff_biz_years].reindex(region_order) - ff_denom.reindex(region_order)[ff_biz_years]
).abs()
print("원자료 사업체수 vs 계산 시트 오차 최대:", ff_diff_denom.max().max())

finance_team_value: data/raw/지표별_측정값/4-1.3. 가족친화 인증기업 비율.xlsx
내부 재계산 오차 최대: 0.0
계산 시트 vs 결과 시트 오차 최대: 0.0
원자료 사업체수 vs 계산 시트 오차 최대: 0.0


In [ ]:
# 18-2. 원자료 개별 기업 목록에서 직접 카운트해서 분자 대조
# 지역명 매핑은 공통 region_map(56457f64 셀)을 재사용한다. 2024년 원자료만
# "대구광역시 수성구"처럼 시군구 상세 주소라, 공백 기준 첫 토큰을 region_map으로 변환한다.


def ff_to_short(value: str) -> str | None:
    if value in region_map:
        return region_map[value]
    first = value.split()[0] if " " in value else value
    return region_map.get(first)


ff_cert = pd.read_excel(family_friendly_path, sheet_name="가족친화 인증기업 현황")
ff_cert.columns = [str(c).strip() for c in ff_cert.columns]
ff_cert["지역"] = ff_cert["지역"].astype(str).str.strip()
ff_cert["지역_단축"] = ff_cert["지역"].apply(ff_to_short)
print("지역 매핑 실패 건수:", ff_cert["지역_단축"].isna().sum())

ff_counts = ff_cert.groupby(["연도", "지역_단축"]).size().unstack("지역_단축")
ff_counts["전국"] = ff_counts.sum(axis=1)
ff_counts = ff_counts.reindex(columns=region_order)

ff_diff_numer = (ff_counts.T[years5] - ff_numer.reindex(region_order)[years5]).abs()
print("원자료 직접 카운트 vs 계산 시트 인증기업 수 오차 최대:", ff_diff_numer.max().max())
print("\n연도별 최대 오차:")
print(ff_diff_numer.max())
print(
    "\n2024년 대구·광주만 각각 59건·20건 차이 (다른 지역·연도는 전부 0.0) - "
    "원인 미상, 이 상세 목록 시트 자체의 누락 가능성"
)